# Tutorial 5 — Structural LTN

Structural LTN extends LTN with **world-indexed reasoning**: every formula can carry
explicit named *structural axes* representing time steps, spatial positions, events, etc.

| Concept | Object | Role |
|---|---|---|
| `Dimension` | logical structural symbol | names a structural domain (e.g. "time") |
| `StructuralVariable` | dimensional variable | ranges over `{0, …, dim.size−1}`, axis name = var name |
| `StructuralRelationSymbol` (declared via `sig.structural_relation`) | structural relation | a fuzzy/boolean matrix over structural indices, used as a guard |
| `forall t: φ` / `exists t: φ` | `Forall` / `Exists` over a structural variable | aggregate the `t` axis with the `StructAgg` operator of `t`'s dimension |
| `phi.free_structural_variables` | `set[str]` of axis names | the structural axes still free (unaggregated) in this formula |


1. `forall t: A(x[t])[t]` — `t` is a structural variable declared over dimension `T`; every structural axis is named by a structural variable, never by a dimension name
2. `x[t]` names the axis of `x`; `A(x[t])[t]` names the predicate's output axis (`[t]` passes the argument axis through, a new name such as `[t1]` creates a new axis)
3. `forall t, t1 | R(t, t1): (A(x[t])[t] -> A(x[t1])[t1])` — guarded structural quantification
4. `x1`, `t1`, `x'`, `t'` — alias variables / StructuralVariables sharing the same grounding

In [ ]:
import torch
import sltn
from sltn import Signature, Interpretation
from sltn.interpretation.operators.aggregator import AggregPMean, AggregPMeanError

torch.manual_seed(0)

## 1 — Building a structural signature

A `Dimension` is a named structural domain declared on the `Signature`.
A `Variable` with `dims=[T]` carries a structural axis in its data tensor
(`[batch, T.size, features]`).  A `Predicate` with `output_dims=[T]` passes
that axis through to the result.

**Grounding.** `StructAgg` (the ∀/∃
aggregation operators for a structural axis) is declared once per
`Dimension` (e.g. `"T"`). Every `StructuralVariable` that ranges over that
dimension — the base variable (`t`), any of its auto-created aliases
(`t1`, `t2`, ...), or an independently declared variable over the same
dimension — automatically inherits it. Grounding can also be set on a
structural variable directly (e.g. `logic["∀,t"] = op`).

**Axis layout.** For a variable `x` of sort `S` (grounded as 1-D vectors) with
`dims=["T"]` where `T` has 5 steps, the grounded tensor for `x` is shaped
`[batch, T.size, features] = [batch, 5, 1]`: a `variable` axis (the batch of individuals), a
`structural` axis (the 5 time steps), and a `domain` axis (the 1 feature of `S`). A predicate
`A` declared with `output_dims=["T"]` preserves that structural axis in its output, so
`A(x[t])[t]` is shaped `[batch, 5, 1]` — a truth value per individual, per time step — until a
structural quantifier (`∀ t: …` / `∃ t: …`) aggregates the `t` axis away.

In [ ]:
sig = Signature("tutorial")
sig.sort("S")                   # sort for individuals
sig.dimension("T")              # temporal dimension, 5 steps
sig.structural_variable("t", "T")

sig.variable("x", "S", dims=["T"])          # x: [batch, t, features]
sig.predicate("A", ["S"], input_dims=["T"], output_dims=["T"])

sig.structural_relation("next", ["T", "T"])          # adjacency relation

print(sig)

## 2 — Interpretation and data groundings

The cell below sets the structural operators with **dictionary-style keys** of the form
`"<quantifier symbol>,<dimension or structural-variable name>"` (e.g. `interp.logic["∀,T"]`,
`interp.logic["∃,t"]`). Aliases (`t`, `t1`, …) resolve to their underlying dimension
automatically, so you don't need to know which name was used to declare the aggregator originally.

In [ ]:
interp = Interpretation(sig)
interp["S"]   = sltn.Type("R1", shape=(1,))
interp["x"]   = torch.rand(4, 5, 1)        # 4 individuals x 5 time steps x 1 feature
interp["A"]   = lambda v: torch.sigmoid(v)

# Structural aggregators are grounded per DIMENSION ("T"), not per variable.
# Every StructuralVariable ranging over T (t, its aliases t1/t2/..., or any
# other variable independently declared over T) inherits this grounding.
interp.logic["∀,T"] = AggregPMeanError(p=2.0)
interp.logic["∃,T"] = AggregPMean(p=2.0)

next_mat = torch.zeros(5, 5)
for i in range(4):
    next_mat[i, i+1] = 1.0
interp["next"] = next_mat

print("x tensor shape:", interp["x"].shape)
print("next matrix:\n", next_mat.numpy())
print("\ninterp.logic['∀,T']:", interp.logic["∀,T"])
print("interp.logic['∃,T']:", interp.logic["∃,T"])

# It is also possible to set the grounding on a STRUCTURAL VARIABLE instead
# (e.g. "t") — this binds the corresponding dimension ("T") anyway, so the
# two forms are interchangeable and stay in sync:
interp.logic["∀,t"] = AggregPMeanError(p=2.0)
print("interp.logic['∀,t'] set → interp.logic['∀,T'] reads it back:", interp.logic["∀,T"])



## 3 — `∀ t: A(x[t])[t]` and `∃ t: A(x[t])[t]`

In `forall t: A(x[t])[t]`, `t` is the `StructuralVariable` declared over dimension `T`.
The formula is an ordinary `Forall` node; since `t` is a structural variable, the `t` axis is aggregated using
whatever `AggregPMeanError` (∀) / `AggregPMean` (∃) is registered for dimension `T` —
looked up from `t`'s dimension, regardless of which variable name is used in the formula.

Note the `__repr__`: `x[t]` shows that `x`'s structural axis is named `t`, and `[t]` after the
predicate call names its structural output axis.

In [ ]:
phi_fa = sig.parse("∀ t: A(x[t])[t]")
phi_ex = sig.parse("∃ t: A(x[t])[t]")
print("∀ repr:", phi_fa) # ∀ t: A(x[t])[t]
print("∃ repr:", phi_ex)

r_fa = interp(phi_fa)
r_ex = interp(phi_ex)
print("\n∀ t A(x[t])[t] shape:", r_fa.value.shape, " (batch=4, 1 feature)")
print("∃ t A(x[t])[t] shape:", r_ex.value.shape)
print("∀ ≤ ∃ everywhere:", (r_fa.value <= r_ex.value + 1e-6).all().item())

In [ ]:
# Every structural axis must be named with a structural variable: leaving an axis
# unnamed, or naming it with the dimension name `T`, is rejected at parse time.
for s in ["A(x)", "A(x[t])", "A(x[T])[T]"]:
    try:
        sig.parse(s)
    except (ValueError, NameError) as e:
        print(f"{s!r} rejected: {e}\n")

## 4 — `A(x[t])[t]` vs `A(x[t1])[t1]` — independent structural axis copies

`A(x[t1])[t1]` reads `x` along an axis named `t1` and passes it through to the output.
`t1` is auto-created as a **StructuralVariable alias** of `t`, over the same dimension `T`.

This lets the same predicate appear under two independent indices:

```
A(x[t])[t] -> A(x[t1])[t1]   # shape [batch, t, t1, 1]
```

`forall t, t1: ...` then aggregates both axes away to `[batch, 1]`.

In [ ]:
r_t = interp(sig.parse("A(x[t])[t]"))
r_t1 = interp(sig.parse("A(x[t1])[t1]"))
print("A(x[t])[t] axes:", r_t.free_structural_variables, " shape:", r_t.value.shape)
print("A(x[t1])[t1] axes:", r_t1.free_structural_variables, " shape:", r_t1.value.shape)

r_impl = interp(sig.parse("A(x[t])[t] -> A(x[t1])[t1]"))
print("\nA(x[t])[t] -> A(x[t1])[t1] shape:", r_impl.value.shape, " (4x5x5x1)")

r_quant = interp(sig.parse("∀ t, t1: (A(x[t])[t] -> A(x[t1])[t1])"))
print("∀ t,t1: ... shape:", r_quant.value.shape, " (4x1)")

In [ ]:
# A(x[t])[t2] is valid: A consumes x's axis t (its input_dims) and emits its output axis
# under the new name t2 (an auto-created alias of t, over T) — same shape, different axis name.
r_t2 = interp(sig.parse("A(x[t])[t2]"))
print("A(x[t])[t2] axes:", r_t2.free_structural_variables, " shape:", r_t2.value.shape)

# Postfix `[axis=idx]` *selection* (not renaming) does bounds-check against the tensor size —
# picking an out-of-range index raises an error at evaluation time.
# Note: a selection must target a structural axis named in the formula, so name
# `A(x[t])`'s output axis first with `[t]`; selecting by the dimension name, as in
# `A(x[t])[t][T=10]`, is rejected at parse time.
try:
    interp(sig.parse("A(x[t])[t][t=10]"))   # dimension T only has 5 steps (indices 0-4)
except IndexError as e:
    print("\nA(x[t])[t][t=10] correctly raises IndexError:", str(e)[:80], "...")

## 5 — `forall t, t1 | R(t, t1): φ` — guarded structural quantification

A structural guard `R(t, t1)` evaluates the structural predicate and uses the result as a
soft (float) or hard (bool) mask in the aggregation.  Pairs where `R = 0` receive zero
weight; pairs where `R = 1` contribute fully.

```
forall t, t1 | next(t, t1): A(x[t])[t] -> A(x[t1])[t1]
```

*"For every consecutive time pair, if A holds at t then A holds at t+1."*

In [ ]:
phi_next = sig.parse("∀ t, t1 | next(t,t1) : (A(x[t])[t] -> A(x[t1])[t1])")
print("repr:", phi_next)

r_next = interp(phi_next)
r_unguarded = interp(sig.parse("∀ t, t1: (A(x[t])[t] -> A(x[t1])[t1])"))
print("\nGuarded values:", r_next.value.squeeze().detach().tolist())
print("Unguarded values:", r_unguarded.value.squeeze().detach().tolist())

## 6 — Variable aliases: `u1` shares the grounding of `u`

`u1`, `u'`, `u0` etc. are **Variable aliases** bound to the same data as `u`
but treated as distinct free variables so formulas range over all pairs.

- `B(u, u1)` → `[4, 4, 1]` (all pairs)
- `B(u, u)`  → `[4, 1]` (diagonalised — same individual)
- `forall u, u1: B(u, u1)` → scalar

In [ ]:
sig.predicate("B", ["S", "S"])
sig.variable("u", "S")
interp["B"] = lambda a, b: torch.sigmoid(a * b)
interp["u"] = torch.rand(4, 1)

print("B(u, u1)       shape:", interp(sig.parse("B(u, u1)")).value.shape, " — all pairs")
print("B(u, u)        shape:", interp(sig.parse("B(u, u)")).value.shape,  " — diagonal")
print("∀ u,u1: B shape:", interp(sig.parse("∀ u, u1: B(u, u1)")).value.shape, " — scalar")

## 7 — Knowledge bases and the training objective

A `KB` groups formulas and aggregates them via `SatAgg`.  Use it as the
training objective: `loss = 1 - interp(phi_kb).value`.
`with_updates(**groundings)` returns a shallow copy with additional overrides.

In [ ]:
phi_kb = sig.parse("{ ∀ x: ∀ t: A(x[t])[t], ∀ x: ∀ t, t1 | next(t, t1): (A(x[t])[t] -> A(x[t1])[t1]) }")
print("KB:", phi_kb)
print("sat:", interp(phi_kb).value.item())

# Typical training loop pattern
# optimizer.zero_grad()
# loss = 1.0 - interp_train(phi_kb).value
# loss.backward(); optimizer.step()

## 8 — `fvs(φ)`: the dimension calculus

`phi.free_structural_variables` gives the set of structural axis *names* still free (unaggregated)
in the result tensor, computed statically after `sig.parse(...)`.

| Formula | `d(φ)` |
|---|---|
| `A(x[t])[t]` | `{'t'}` |
| `forall t: A(x[t])[t]` | `∅` |
| `forall t, t1: A(x[t])[t] -> A(x[t1])[t1]` | `∅` |

In [ ]:
for s in ["A(x[t])[t]", "∀ t: A(x[t])[t]", "∀ t, t1: (A(x[t])[t] -> A(x[t1])[t1])", "∀ t: (A(x[t])[t] -> A(x[t1])[t1])"]:
    phi = sig.parse(s)
    print(f"d({s!r})  →  {phi.free_structural_variables}")

## 9 — Selecting an index

A selection `[day=k]` picks index `k` of the structural axis `day` (a structural variable over
dimension `Day`) and removes the axis.

In [ ]:
sig_labeled = Signature("labeled_demo")
sig_labeled.sort("S")
day_dim = sig_labeled.dimension("Day")
sig_labeled.structural_variable("day", "Day")

x_lab = sig_labeled.variable("x", "S", dims=["Day"])
P_lab = sig_labeled.predicate("P", ["S"], input_dims=["Day"], output_dims=["Day"])

interp_lab = Interpretation(sig_labeled)
interp_lab["S"] = sltn.Type("R1", shape=(1,))
interp_lab["x"] = torch.rand(3, 5, 1)   # 3 individuals x 5 labelled days x 1 feature
interp_lab["P"] = lambda v: torch.sigmoid(v)

interp_lab("P(x[day])[day][day=0]")